In [ ]:
from _settings import *    
import pandas as pd    
print(TOPDIR)    
#delimiter je SK, bad_lines vypíše a skipne, index_col=False - neberie prvý stĺpec ako index    
# df = pd.read_csv(VRTCSVQGIS, delimiter=';', on_bad_lines='warn', index_col=False)    
df = pd.read_csv(VRTCSVQGIS, delimiter=';', on_bad_lines='warn', index_col=False)    
pd.set_option('display.max_colwidth', 1000)    
pd.set_option('display.max_rows', None)    
#tu je problem, lebo neviem zobrazit kompletny riadok 10053    
dupl = df[df.duplicated(subset=["Vrt", "Uloha", "JTSKX", "JTSKY"], keep=False)].sort_values(by=["Vrt","File"]).loc[:,["Vrt", "File", "JTSKX", "JTSKY"]]    
print(f'all in df={df.shape[0]}, dupl in df={dupl.shape[0]}')    
count_df445 = df[df.File.str.contains('_445')].shape[0]    
count_dupl445 = dupl[dupl.File.str.contains('_445')].shape[0]    
print(f"all 445={count_df445} - dupl 445={count_dupl445} = {count_df445-count_dupl445}")    
nodupl = df[~ df.duplicated(subset=["Vrt", "Uloha", "JTSKX", "JTSKY"], keep=False)].sort_values(by=["Vrt", "File"]).loc[:,["Vrt", "File", "JTSKX", "JTSKY"]]    
print(nodupl.shape[0])    
nodupl[nodupl.File.str.contains('_445')]    


In [ ]:
def clean_duplicates1(df):
    
    df_nodup = df[~ df.duplicated(subset=["Vrt", "JTSKX", "JTSKY"], keep=False)]
    df_dup = df.sort_values(by='File',ascending=True)
    df_dup = df_dup[~df_dup.duplicated(subset=["Vrt",  "JTSKX", "JTSKY"], keep='last')]
    print(f"original:{df.shape} nodup:{df_nodup.shape} reduced{df_dup.shape}")
    # return pd.concat([df_nodup, df_dup])
    return df_dup

def clean_duplicates(df):
    df_dup = df.sort_values(by='File',ascending=True).drop_duplicates(subset=["Vrt", "JTSKX", "JTSKY"], keep='last')
    print(f"original:{df.shape}  reduced{df_dup.shape}")
    return df_dup

def clean_coordinates(df):
    # return df[(df.JTSKY > 595000) | (df.JTSKY < 160000) | (df.JTSKX> 1340000) | (df.JTSKX < 1128000) ] # bad values
    return df[(df.JTSKY < 595000) & (df.JTSKY > 160000) & (df.JTSKX < 1340000) & (df.JTSKX > 1128000) ] # good values
    # pass

def to_num(df, cols):
    '''prevedie stlpce cols na numeric'''
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors='coerce', dtype_backend='numpy_nullable').astype('Float32')
        #memory usage 60347 vs 80255 bytes
        #df[col] = pd.to_numeric(df[col], errors='coerce', dtype_backend='pyarrow' ).astype(pd.ArrowDtype(pa.float64())) 
    return df

def apply_swap(x, y):
    if x < y:
        return y, x
    else:
        return y, x
    



    

df = to_num(df, ['JTSKX', 'JTSKY'])
dc1=df.copy()
# dc1=dc1[dc1.Vrt=='VL-1']
for _ in range(10):
    dc1 = clean_duplicates1(dc1)
dc1.tail()

dfc = clean_coordinates(dc1)
dfc.describe()

dfc['JTSKCoor'] =  dfc.apply(lambda x: apply_swap(x.JTSKX, x.JTSKY), axis=1 )
# print(dfc[:,'JTSKX','JTSKY'])
dfc.info()
dfcswap = clean_coordinates(dfc)
dfcswap.info()
# dc1 = dc1[dc1.duplicated(subset=["Vrt", "Uloha", "JTSKX", "JTSKY"], keep=False)].sort_values(by='File',ascending=True)
# dc1[dc1.Vrt=='VL-1'].shape

# df[(df.JTSKX < 0) | (df.JTSKY < 0)]

dfcswap.JTSKCoor.head()

x = dfcswap.iloc[1,30]
type(x)
x[0]
xxx = dfcswap.JTSKCoor.to_list()
xxx



In [ ]:

df1 = to_num(df,['JTSKX', 'JTSKY'])
df1.info()

print(df1.JTSKX.min(),df1.JTSKX.max(), df1.JTSKX.mean())
pd.DataFrame(df1.JTSKX.sort_values().unique()).head(20)



In [ ]:
df2=df[1:100]
df2=df2.sort_values(by='File',ascending=True)
df2dupl = df2[df2.duplicated(subset=["Vrt", "Uloha", "JTSKX", "JTSKY"], keep='last')].sort_values(by=["Vrt","File"]).loc[:,["Vrt", "File", "JTSKX", "JTSKY"]]    
df2dupl

In [ ]:
def swap(x, y):
    return [y, x]

x = 5
y = 6

x, y = swap(x, y)
print (x, y)
x, y = swap(x, y)
print (x, y)

In [10]:
from _utils import *
import re
# flist = dirEntries(r'C:\\\\Shares\vrty\vrty3\python\OK-CSV', True, 'OK')
flist = dirEntries(r'C:\Users\envigeo\program\python\vrty\OK-CSV', True, 'OK')
# Subdirs(r'C:\Shares\vrty\vrty3\python\ErrorsOK', True)
# pattern = r"Vrty_202509"  # Use r"" for raw strings to handle backslashes
# replacement = r"ErrorsOK"
pattern = r"c:\\Shares\\vrty\\vrty3\\python\\Vrty_202509"
replacement = r"C:\\Users\\envigeo\\program\\python\\vrty\\Vrty_202509"

for input_file in flist:
    # 2. Load the file
    with open(input_file, "r", encoding='cp1250') as file:
        content = file.read()

    # 3. Perform regex substitution
    modified_content = re.sub(pattern, replacement, content)

    # 4. Save under a different name
    print(input_file)
    # output_file = re.sub(input_file, r'.OK', '')
    output_file = input_file[:-3]
    print(output_file)
    with open(output_file, "w", encoding='cp1250') as file:
        file.write(modified_content)

    print(f"File saved as {output_file}")



C:\Users\envigeo\program\python\vrty\OK-CSV\10031\vrty.csv.OK
C:\Users\envigeo\program\python\vrty\OK-CSV\10031\vrty.csv
File saved as C:\Users\envigeo\program\python\vrty\OK-CSV\10031\vrty.csv
C:\Users\envigeo\program\python\vrty\OK-CSV\11351\vrty.csv.OK
C:\Users\envigeo\program\python\vrty\OK-CSV\11351\vrty.csv
File saved as C:\Users\envigeo\program\python\vrty\OK-CSV\11351\vrty.csv
C:\Users\envigeo\program\python\vrty\OK-CSV\11576\vrty.csv.OK
C:\Users\envigeo\program\python\vrty\OK-CSV\11576\vrty.csv
File saved as C:\Users\envigeo\program\python\vrty\OK-CSV\11576\vrty.csv
C:\Users\envigeo\program\python\vrty\OK-CSV\11598\prerobene\vrty.csv.OK
C:\Users\envigeo\program\python\vrty\OK-CSV\11598\prerobene\vrty.csv
File saved as C:\Users\envigeo\program\python\vrty\OK-CSV\11598\prerobene\vrty.csv
C:\Users\envigeo\program\python\vrty\OK-CSV\11598\vrty.csv.OK
C:\Users\envigeo\program\python\vrty\OK-CSV\11598\vrty.csv
File saved as C:\Users\envigeo\program\python\vrty\OK-CSV\11598\vrty.csv
